# TripAdvisor Information Retrieval Project
**Project 1 - Information Retrieval & NLP**

Alvaro SERERO, Leo WINTER

ESILV A4 DIA6

## 1. Setup & Imports

In [13]:
# !pip install rank-bm25 scikit-learn spacy pandas numpy tqdm
# !python -m spacy download en_core_web_sm

import ast
import gc
import re
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DATA_PATH    = Path("TripAdvisorTrainingDataProject1")
RANDOM_STATE = 42

# Columns used for labels (for evaluation only)
LABEL_COLUMNS   = ["typeR"]
REVIEWS_COLUMNS = ["idplace", "review", "langue"]

# Load English spaCy model (we only use English reviews)
nlp = spacy.load("en_core_web_sm")

## 2. Load Data

In [15]:
attraction_sub_categorie = pd.read_csv(DATA_PATH / "AttractionSubCategorie.csv")
attraction_sub_type = pd.read_csv(DATA_PATH / "AttractionSubType.csv")
cuisine_df = pd.read_csv(DATA_PATH / "cuisine.csv")
dietary_restrictions_df = pd.read_csv(DATA_PATH / "dietary_restrictions.csv")
restaurant_type_df = pd.read_csv(DATA_PATH / "restaurantType.csv")
reviews = pd.read_csv(DATA_PATH / "reviews83325.csv", low_memory=False) 
trip_advisor = pd.read_csv(DATA_PATH / "Tripadvisor.csv", low_memory=False) 

print(f"reviews: {reviews.shape}")
print(f"trip_advisor: {trip_advisor.shape}")
print("\ntrip_advisor columns:", trip_advisor.columns.tolist())

reviews: (340385, 21)
trip_advisor: (3761, 60)

trip_advisor columns: ['id', 'idTrip', 'fromId', 'nom', 'url', 'rating', 'nbAvis', 'nbAvisRecupere', 'latitude', 'longitude', 'typeR', 'adresse', 'priceRange', 'closed', 'hotelType', 'hotelStyle', 'hotelStars', 'hotelRoomNumber', 'hotelNoteEmplacement', 'hotelNoteProprete', 'hotelNoteService', 'HotelNoteQualitePrix', 'hoteldistance', 'hotelbearing', 'restaurantTypeCuisine', 'restaurantDietaryRestrictions', 'restaurantMeals', 'restaurantFeatures', 'restaurantNoteCuisine', 'restaurantNoteService', 'restaurantNoteQualitePrix', 'restaurantNoteAmbiance', 'activiteType', 'activiteSubCategorie', 'activiteSubType', 'website', 'nbScanReview', 'dateLastScanReviews', 'shape_gid', 'gadm36_gid', 'hotelprice', 'hotelBookingID', 'restaurantSubcategory', 'restaurantType', 'ap_additional_info', 'ap_age_band_list', 'ap_attraction_ids', 'ap_booking_question_list', 'ap_bubble_rating_integer', 'ap_duration', 'ap_exclusion', 'ap_inclusions', 'ap_introduction',

## 3. Exploratory Data Analysis

In [16]:
print("Place type distribution (typeR):")
print(trip_advisor["typeR"].value_counts())
# H = Hotel | R = Restaurant | A = Attraction | AP = Attraction Product

Place type distribution (typeR):
typeR
AP    2669
R      623
A      405
H       64
Name: count, dtype: int64


In [17]:
print("\nReview language distribution (top 10):")
print(reviews["langue"].value_counts(normalize=True).mul(100).round(2).head(10))


Review language distribution (top 10):
langue
en    44.97
fr    29.11
it     6.73
es     6.40
pt     5.73
ru     1.60
de     1.54
ja     1.21
nl     0.83
ko     0.33
Name: proportion, dtype: float64


In [6]:
def add_name_from_id(dataset_input, input_column_name, dataset_id):
    """
    Map a column of comma-separated IDs to human-readable names.
    Drops the original ID column; adds a new <col>_name column.
    Always called on a copy — never mutates the source DataFrame.
    """
    if input_column_name not in dataset_input.columns:
        print(f"Column '{input_column_name}' not found — skipping.")
        return dataset_input

    dataset_id = dataset_id.copy()
    dataset_id["id"] = dataset_id["id"].astype(str)
    mapping_id = dataset_id.set_index("id")["name"].to_dict()

    def map_ids(id_string):
        if pd.isna(id_string):
            return np.nan
        parts = str(id_string).split(",")
        return ", ".join(mapping_id.get(p.strip(), "Unknown") for p in parts)

    out = dataset_input.copy()
    out[f"{input_column_name}_name"] = out[input_column_name].apply(map_ids)
    return out.drop(columns=[input_column_name])


# Build a named copy for display only — trip_advisor stays untouched
trip_advisor_named = trip_advisor.copy()
trip_advisor_named = add_name_from_id(trip_advisor_named, "activiteSubCategorie", attraction_sub_categorie)
trip_advisor_named = add_name_from_id(trip_advisor_named, "activiteSubType", attraction_sub_type)
trip_advisor_named = add_name_from_id(trip_advisor_named, "restaurantTypeCuisine", cuisine_df)
trip_advisor_named = add_name_from_id(trip_advisor_named, "restaurantDietaryRestrictions", dietary_restrictions_df)
trip_advisor_named = add_name_from_id(trip_advisor_named, "restaurantType", restaurant_type_df)

print("Sample (human-readable categories):")
print(trip_advisor_named[["id", "nom", "typeR",
                           "activiteSubCategorie_name", "restaurantType_name"]].head(5))

Sample (human-readable categories):
       id                       nom typeR  \
0  188467          Place des Vosges     A   
1  188468  Rue des Francs Bourgeois     A   
2  188470        Village Saint-Paul     A   
3  188471          Au Passe-partout     A   
4  188472     Cloître des Billettes     A   

             activiteSubCategorie_name restaurantType_name  
0                   Sites touristiques                 NaN  
1                   Sites touristiques                 NaN  
2  Shopping, Sites touristiques, Autre                 NaN  
3                             Shopping                 NaN  
4                   Sites touristiques                 NaN  


In [7]:
print("ActiviteSubCategorie :")
print("     Unique values :",trip_advisor['activiteSubCategorie'].nunique())
print("     Nan values",trip_advisor['activiteSubCategorie'].isna().sum())
print("     Total values :",trip_advisor['activiteSubCategorie'].count())
print("     id type :", trip_advisor['activiteSubCategorie'].dtype)

print("\nActiviteSubType :")
print("     Unique values :",trip_advisor['activiteSubType'].nunique())
print("     Nan values",trip_advisor['activiteSubType'].isna().sum())
print("     Total values :",trip_advisor['activiteSubType'].count())
print("     id type :", trip_advisor['activiteSubType'].dtype)

print("\nRestaurantTypeCuisine :")
print("     Unique values :",trip_advisor['restaurantTypeCuisine'].nunique())
print("     Nan values",trip_advisor['restaurantTypeCuisine'].isna().sum())
print("     Total values :",trip_advisor['restaurantTypeCuisine'].count())
print("     id type :", trip_advisor['restaurantTypeCuisine'].dtype)

print("\nRestaurantDietaryRestrictions :")
print("     Unique values :",trip_advisor['restaurantDietaryRestrictions'].nunique())
print("     Nan values",trip_advisor['restaurantDietaryRestrictions'].isna().sum())
print("     Total values :",trip_advisor['restaurantDietaryRestrictions'].count())
print("     id type :", trip_advisor['restaurantDietaryRestrictions'].dtype)

print("\nRestaurantType :")
print("     Unique values :",trip_advisor['restaurantType'].nunique())
print("     Nan values",trip_advisor['restaurantType'].isna().sum())
print("     Total values :",trip_advisor['restaurantType'].count())
print("     id type :", trip_advisor['restaurantType'].dtype)

ActiviteSubCategorie :
     Unique values : 45
     Nan values 3356
     Total values : 405
     id type : object

ActiviteSubType :
     Unique values : 124
     Nan values 3356
     Total values : 405
     id type : object

RestaurantTypeCuisine :
     Unique values : 245
     Nan values 3243
     Total values : 518
     id type : object

RestaurantDietaryRestrictions :
     Unique values : 16
     Nan values 3529
     Total values : 232
     id type : object

RestaurantType :
     Unique values : 14
     Nan values 3138
     Total values : 623
     id type : object


## 4. Preprocessing

- Keep English reviews only
- Drop rows with missing reviews
- Lemmatize with spaCy: lowercase, alphabetic tokens only, stopwords removed

In [22]:
reviews_en = reviews[reviews["langue"] == "en"][REVIEWS_COLUMNS].copy()
reviews_en = reviews_en.dropna(subset=["review"]).reset_index(drop=True)
print(f"English reviews: {len(reviews_en):,} / {len(reviews):,} total")

English reviews: 153,071 / 340,385 total


In [ ]:
def clean_text_spacy(texts: list) -> list:
    """
    Lemmatize a batch of texts using spaCy.
    Keeps only alphabetic, non-stop tokens longer than 2 characters.
    Uses nlp.pipe() for efficiency instead of processing one by one.
    """
    results = []
    for doc in nlp.pipe(texts, batch_size=256, disable=["parser", "ner"]):
        lemmas = [
            t.lemma_.lower()
            for t in doc
            if t.is_alpha and not t.is_stop and len(t) > 2
        ]
        results.append(" ".join(lemmas))
    return results


reviews_en["clean_text"] = clean_text_spacy(reviews_en["review"].tolist())

print("Sample input:", reviews_en["review"].iloc[0])
print("Sample output:", reviews_en["clean_text"].iloc[0])

Sample input : Personally I think it is the most beautiful square of Paris. Well maintained and the area around it gives you opportunities to grab a bite to eat as well.
Sample output: personally think beautiful square paris maintain area give opportunity grab bite eat


## 5. Build Place Documents

Each document contains all cleaned reviews of a place concatenated.

We also apply a cap to reduce the size imbalance between popular and less popular places.

In [24]:
MAX_REVIEWS_PER_PLACE = 50  # set to None to use all reviews

def aggregate_reviews(group):
    texts = group["clean_text"].tolist()
    if MAX_REVIEWS_PER_PLACE:
        texts = texts[:MAX_REVIEWS_PER_PLACE]
    return " ".join(texts)


print("Aggregating reviews by place...")
place_docs = (
    reviews_en
    .groupby("idplace", group_keys=False)
    .apply(aggregate_reviews)
    .reset_index()
    .rename(columns={"idplace": "place_id", 0: "document"})
)
print(f"Places with ≥1 English review: {len(place_docs)}")

# Merge evaluation metadata — keep only columns that exist in this dataset
# Note: cuisine is stored as 'restaurantTypeCuisine' in Tripadvisor.csv (not 'cuisine')
META_COLS = ["id", "typeR", "activiteSubCategorie", "activiteSubType",
             "restaurantType", "restaurantTypeCuisine", "priceRange"]
meta_cols_present = [c for c in META_COLS if c in trip_advisor.columns]

place_docs = place_docs.merge(
    trip_advisor[meta_cols_present].rename(columns={"id": "place_id"}),
    on="place_id",
    how="left",
)
print(f"After metadata merge: {len(place_docs)} places")
print(place_docs["typeR"].value_counts())

Aggregating reviews by place...
Places with ≥1 English review: 1835
After metadata merge: 1835 places
typeR
AP    989
R     538
A     252
H      56
Name: count, dtype: int64


/var/folders/cr/3y3t2vr97xsf_f_y_xhw2jv00000gn/T/ipykernel_55264/3467550162.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(aggregate_reviews)


## 6. Train / Test Split

50% **train**: used as queries  
50% **test**: used as candidates to retrieve from

In [25]:
train_df, test_df = train_test_split(
    place_docs, test_size=0.5, random_state=RANDOM_STATE, shuffle=True
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train (queries):    {len(train_df)}")
print(f"Test  (candidates): {len(test_df)}")

Train (queries):    917
Test  (candidates): 918


## 7. Models

### Baseline Model: BM25

In [26]:
# Building BM25 index on test candidates
test_tokenized = [doc.split() for doc in test_df["document"]]
bm25_index = BM25Okapi(test_tokenized)

def retrieve_bm25(query_doc: str) -> np.ndarray:
    """Return test indices ranked by BM25 score (highest first)."""
    scores = bm25_index.get_scores(query_doc.split())
    return np.argsort(scores)[::-1]

### Improved Model: TF-IDF with Cosine Similarity

In [27]:
# Fitting TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=20_000, ngram_range=(1, 2), sublinear_tf=True)

# Fit on all documents so vocabulary covers both train and test
tfidf_vectorizer.fit(place_docs["document"])

train_matrix = tfidf_vectorizer.transform(train_df["document"])  # query vectors
test_matrix  = tfidf_vectorizer.transform(test_df["document"])   # candidate vectors

print(f"Train matrix: {train_matrix.shape}")
print(f"Test  matrix: {test_matrix.shape}")
gc.collect()

Train matrix: (917, 20000)
Test  matrix: (918, 20000)


0

In [28]:
def get_ranked_indices_batched(query_matrix, candidate_matrix, batch_size: int = 200) -> np.ndarray:
    """
    For each query, compute cosine similarity against all candidates and
    return full rankings (highest score first).
    Processes queries in batches to avoid memory spikes.

    Returns: (n_queries, n_candidates) array of ranked candidate indices.
    """
    n_queries  = query_matrix.shape[0]
    all_ranked = []

    for i in tqdm(range(0, n_queries, batch_size), desc="TF-IDF ranking"):
        batch      = query_matrix[i : i + batch_size]
        sims_batch = (batch @ candidate_matrix.T).toarray()   # (batch, n_candidates)
        ranked     = np.argsort(sims_batch, axis=1)[:, ::-1]  # descending
        all_ranked.append(ranked)

    return np.vstack(all_ranked)


# Computing TF-IDF rankings
tfidf_rankings = get_ranked_indices_batched(train_matrix, test_matrix)
print(f"Rankings matrix: {tfidf_rankings.shape}")
gc.collect()

TF-IDF ranking: 100%|██████████| 5/5 [00:00<00:00, 55.16it/s]

Rankings matrix: (917, 918)


0

## 8. Evaluation

In [29]:
def parse_id_list(val) -> set:
    """Parse a comma-separated or JSON-list ID field into a set of strings."""
    if pd.isna(val) or str(val).strip() in ("", "[]", "nan"):
        return set()
    if isinstance(val, (list, set)):
        return {str(v).strip() for v in val}
    try:
        parsed = ast.literal_eval(str(val))
        if isinstance(parsed, list):
            return {str(v).strip() for v in parsed}
    except Exception:
        pass
    return {s.strip() for s in str(val).split(",") if s.strip()}


def get_L2_label(row) -> set:
    """
    Build the set of L2 category identifiers for a place, depending on its type.
    Uses raw ID columns (not the _name columns).
    """
    type_r = row.get("typeR", "")
    if type_r in ("A", "AP"):
        return parse_id_list(row.get("activiteSubCategorie")) | parse_id_list(row.get("activiteSubType"))
    elif type_r == "R":
        return parse_id_list(row.get("restaurantType")) | parse_id_list(row.get("restaurantTypeCuisine"))
    elif type_r == "H":
        pr = row.get("priceRange")
        return {str(pr)} if pr and not pd.isna(pr) else set()
    return set()


# Pre-compute L2 labels for all places
train_df["L2_label"] = train_df.apply(get_L2_label, axis=1)
test_df["L2_label"]  = test_df.apply(get_L2_label,  axis=1)
print("L2 labels computed.")

L2 labels computed.


In [30]:
def ranking_error_L1(train_df: pd.DataFrame, test_df: pd.DataFrame, rankings: np.ndarray) -> float:
    """
    Level-1 Ranking Error: for each query find the position of the first candidate
    with the same typeR.  Error = position index (0 = perfect match at rank 1).
    Queries with no matching type in the test set are skipped (undefined).
    """
    errors    = []
    undefined = 0
    test_types = test_df["typeR"].to_numpy()

    for q_idx, q_row in tqdm(train_df.iterrows(), total=len(train_df), desc="L1 eval"):
        q_type = q_row["typeR"]
        if not np.any(test_types == q_type):
            undefined += 1
            continue
        for position, cand_idx in enumerate(rankings[q_idx]):
            if test_types[cand_idx] == q_type:
                errors.append(position)
                break

    mean_err = float(np.mean(errors)) if errors else float("inf")
    print(f"  Evaluated: {len(errors)} queries | Undefined: {undefined}")
    print(f"  Mean Ranking Error L1: {mean_err:.4f}")
    return mean_err


def ranking_error_L2(train_df: pd.DataFrame, test_df: pd.DataFrame, rankings: np.ndarray) -> float:
    """
    Level-2 Ranking Error: same as L1 but a positive match requires same typeR
    AND at least one L2 category in common (partial match counts).
    """
    errors      = []
    undefined   = 0
    test_types  = test_df["typeR"].to_numpy()
    test_labels = test_df["L2_label"].to_numpy()

    for q_idx, q_row in tqdm(train_df.iterrows(), total=len(train_df), desc="L2 eval"):
        q_type  = q_row["typeR"]
        q_label = q_row["L2_label"]

        if not q_label:
            undefined += 1
            continue

        # Skip if no matching candidate exists in the test set
        has_match = any(
            test_types[i] == q_type and bool(test_labels[i] & q_label)
            for i in range(len(test_df))
        )
        if not has_match:
            undefined += 1
            continue

        for position, cand_idx in enumerate(rankings[q_idx]):
            if test_types[cand_idx] == q_type and bool(test_labels[cand_idx] & q_label):
                errors.append(position)
                break

    mean_err = float(np.mean(errors)) if errors else float("inf")
    print(f"  Evaluated: {len(errors)} queries | Undefined: {undefined}")
    print(f"  Mean Ranking Error L2: {mean_err:.4f}")
    return mean_err

### BM25 Evaluation

In [33]:
# Computing BM25 rankings for all queries
bm25_rankings = np.array([
    retrieve_bm25(row["document"])
    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="BM25")
])

print("\nBM25 – Level 1")
bm25_L1 = ranking_error_L1(train_df, test_df, bm25_rankings)

print("\nBM25 – Level 2")
bm25_L2 = ranking_error_L2(train_df, test_df, bm25_rankings)

BM25: 100%|██████████| 917/917 [00:50<00:00, 18.26it/s]



BM25 – Level 1


L1 eval: 100%|██████████| 917/917 [00:00<00:00, 39981.05it/s]


  Evaluated: 917 queries | Undefined: 0
  Mean Ranking Error L1: 0.7764

BM25 – Level 2


L2 eval: 100%|██████████| 917/917 [00:00<00:00, 83774.62it/s]

  Evaluated: 414 queries | Undefined: 503
  Mean Ranking Error L2: 5.4541


### TF-IDF Evaluation

In [34]:
print("TF-IDF – Level 1")
tfidf_L1 = ranking_error_L1(train_df, test_df, tfidf_rankings)

print("\nTF-IDF – Level 2")
tfidf_L2 = ranking_error_L2(train_df, test_df, tfidf_rankings)

TF-IDF – Level 1


L1 eval: 100%|██████████| 917/917 [00:00<00:00, 38363.56it/s]


  Evaluated: 917 queries | Undefined: 0
  Mean Ranking Error L1: 0.7743

TF-IDF – Level 2


L2 eval: 100%|██████████| 917/917 [00:00<00:00, 35959.36it/s]

  Evaluated: 414 queries | Undefined: 503
  Mean Ranking Error L2: 2.6594
